# MEMB impurity model

Этот Jupyter notebook реализует демонстрацию газообмена в топливных элементах PEMFC (Polymer Electrolyte Membrane Fuel Cells).
Модель реализована на базе реконструкции уравнений аналогичных квазистационарной модели топливного элемента в среде моделирования Simcenter Amesim.
В модели определяются потоки O₂/N₂ к аноду, H₂ к катоду, потребление H₂, перенос H₂O, λ_memb, φ_w, накопление примесей со временем, расходы в системе рециркуляции анода.

**Модель предназначена для учебных целей**

## Основные компоненты модели:

1. **Физические параметры мембраны**: 
   - Количество ячеек БТЭ, $шт$
   - Активная площадь одной ячейки БТЭ, $см^2$
   - Толщина мембраны, $мм$
   - Эквивалентная масса иономера, $г/моль$
   - Плотность материала мембраны, $кг/м^3$.

2. **Транспортные явления**:
   - Диффузия газов (O₂, N₂, H₂) через мембрану (crossover)
   - Диффузия воды (Fick's law)
   - Электроосмотический перенос воды (drag)

3. **Ключевые уравнения** (по Springer et al. 1991, Zawodzinski et al. 1993):
   - Давление насыщения воды: Antoine equation
   - Активность воды и содержание $λ_mem(a_w)$
   - Коэффициент $drag n_drag(λ_avg)$
   - Диффузия воды $D_w(λ_avg, T)$


## 1. Базовые параметры БТЭ, состояния анода и катода Amesim и физические константы

Вынесены в начало, чтобы можно было быстро менять конфигурацию без поиска по всему ноутбуку.
Единицы измерения сохранены рядом с каждой переменной.


In [1]:
# -----------------------------
# ПАРАМЕТРЫ БТЭ
# -----------------------------

# Количество ячеек БТЭ, шт.
Ncell = 250

# Активная площадь одной ячейки БТЭ, см^2.
Scell = 417.74

# Толщина мембраны, мм.
lmemb = 0.125

# Эквивалентная масса иономера, г/моль.
# По смыслу: масса сухого полимера на 1 моль сульфогрупп SO3-.
EW = 1100.0

# Плотность материала мембраны, кг/м^3.
rho_memb = 2000.0

# -----------------------------
# ВХОДНЫЕ СОСТОЯНИЯ AMESIM
# -----------------------------

# Анод: температура смеси, К.
ANODE_T_K = 301.582

# Анод: абсолютное давление смеси, Па.
ANODE_P_PA = 199404.0

# Анод: молярные доли компонентов смеси.
ANODE_X_O2 = 0.00
ANODE_X_N2 = 0.0235389
ANODE_X_H2O = 0.0188723
ANODE_X_H2 = 1 - ANODE_X_O2 - ANODE_X_N2 - ANODE_X_H2O

# Катод: температура смеси, К.
CATHODE_T_K = 353.0

# Катод: абсолютное давление смеси, Па.
CATHODE_P_PA = ANODE_P_PA + 10e3

# Катод: молярные доли компонентов смеси.
# Здесь по умолчанию задан демонстрационный состав катодной смеси. Для примера влажного воздуха можно использовать значения из комментариев справа.
CATHODE_X_O2 = 0.0 # 0.1250
CATHODE_X_N2 = 0.7 # 0.4700
CATHODE_X_H2O = 0.3 # 0.4050
CATHODE_X_H2 = 1 - CATHODE_X_O2 - CATHODE_X_N2 - CATHODE_X_H2O  # 0.0

# -----------------------------
# РАСЧЁТНЫЙ ТОК
# -----------------------------

# Ток БТЭ для демонстрационного расчёта, А.
CURRENT_A = 300.0

# -----------------------------
# ПАРАМЕТРЫ ЗАКРЫТОЙ РЕЦИРКУЛЯЦИИ
# -----------------------------

# Стехиометрическое число на входе в БТЭ.
# По модели: dm1 = lambda * dm_H2.
ANODE_STOICH_LAMBDA = 1.5

# Коэффициент после влагоотделителя, %.
# Используется формула humid_out = (eff/100) * humid_in.
# 100% означает ту же относительную влажность, 0% означает полное осушение.
WATER_SEPARATOR_EFF_PERCENT = 93.0

# Горизонт накопления примесей, с.
ACCUMULATION_TIME_S = 100.0

# -----------------------------
# ФУНДАМЕНТАЛЬНЫЕ КОНСТАНТЫ
# -----------------------------

# Постоянная Фарадея, Кл/моль.
FARADAY = 96485.3415

# Универсальная газовая постоянная, Дж/(моль·К).
R_UNIV = 8.3144

# Молярные массы компонентов, г/моль.
M_H2_G_MOL = 2.016
M_N2_G_MOL = 28.01
M_H2O_G_MOL = 18.015
M_O2_G_MOL = 31.998


## 2. Импорты

- `dataclass` для структур данных
- `math` для элементарных функций
- `pandas` для удобного просмотра результата


In [2]:
from dataclasses import dataclass, asdict
from math import exp, log10
from typing import Dict

import pandas as pd


## 3. Структуры данных модели

- `BTEParams` — параметры БТЭ и мембраны
- `GasState` — состояние газовой смеси на одной стороне мембраны
- `SC3Result` — результат расчёта потоков и внутренних переменных

Внутри `BTEParams` вычисляются свойства:
- суммарная активная площадь БТЭ
- толщина мембраны в СИ
- концентрация фиксированных сульфогрупп в мембране


In [3]:
@dataclass
class BTEParams:
    # Количество ячеек БТЭ, шт.
    ncell: int = Ncell

    # Площадь одной ячейки БТЭ, см^2.
    scell_cm2: float = Scell

    # Толщина мембраны, мм.
    lmemb_mm: float = lmemb

    # Эквивалентная масса иономера, г/моль.
    ew_g_per_mol: float = EW

    # Плотность мембраны, кг/м^3.
    rho_memb_kg_m3: float = rho_memb

    @property
    def area_m2_total(self) -> float:
        # Суммарная активная площадь БТЭ, м^2.
        return (self.scell_cm2 / 10000.0) * self.ncell

    @property
    def thickness_m(self) -> float:
        # Толщина мембраны, м.
        return self.lmemb_mm / 1000.0

    @property
    def c_so3_mol_m3(self) -> float:
        # Концентрация неподвижных сульфогрупп SO3- в мембране, моль/м^3.
        return self.rho_memb_kg_m3 / (self.ew_g_per_mol / 1000.0)


@dataclass
class GasState:
    # Температура газовой смеси, К.
    temperature_K: float

    # Полное давление смеси, Па.
    pressure_Pa: float

    # Молярные доли компонентов.
    x_h2: float
    x_n2: float
    x_h2o: float
    x_o2: float

    @property
    def p_h2(self) -> float:
        # Парциальное давление водорода, Па.
        return self.pressure_Pa * self.x_h2

    @property
    def p_n2(self) -> float:
        # Парциальное давление азота, Па.
        return self.pressure_Pa * self.x_n2

    @property
    def p_h2o(self) -> float:
        # Парциальное давление водяного пара, Па.
        return self.pressure_Pa * self.x_h2o

    @property
    def p_o2(self) -> float:
        # Парциальное давление кислорода, Па.
        return self.pressure_Pa * self.x_o2

    def validate(self) -> None:
        # Проверка, что сумма молярных долей близка к 1.
        total = self.x_h2 + self.x_n2 + self.x_h2o + self.x_o2
        if abs(total - 1.0) > 1e-6:
            raise ValueError(f"Сумма молярных долей должна быть 1.0, сейчас {total}")


@dataclass
class SC3Result:
    # Газовый crossover через мембрану, моль/с.
    n_o2_to_anode_mol_s: float
    n_n2_to_anode_mol_s: float
    n_h2_to_cathode_mol_s: float

    # Потоки воды через мембрану, моль/с.
    n_h2o_diff_to_anode_mol_s: float
    n_h2o_drag_to_cathode_mol_s: float
    n_h2o_net_to_anode_mol_s: float

    # Реакционный расход водорода на аноде, моль/с.
    n_h2_consumption_mol_s: float

    # Внутренние расчётные величины.
    lambda_membrane_anode: float
    lambda_membrane_cathode: float
    phi_w_anode: float
    phi_w_cathode: float
    phi_w_avg: float
    a_h2o_anode: float
    a_h2o_cathode: float
    p_sat_anode_Pa: float
    p_sat_cathode_Pa: float
    n_drag: float


def result_to_dataframe(result: SC3Result) -> pd.DataFrame:
    # Перевод результата в таблицу для удобного просмотра в ноутбуке.
    return pd.DataFrame(
        {"variable": list(asdict(result).keys()), "value": list(asdict(result).values())}
    )


## 4. Вспомогательные функции мембраны

Зависимости, которые использует `MEMB`:

- давление насыщенного пара воды
- активность воды
- гидратация мембраны
- объемная доля воды в мембране
- коэффициент electro-osmotic drag
- коэффициент диффузии воды в мембране

## Физический смысл уравнений этого блока

### 4.1. Давление насыщенного пара воды

Функция `water_saturation_pressure_Pa(T_K)` возвращает давление насыщенного водяного пара:

$$
p_{\mathrm{sat}} = p_{\mathrm{sat}}(T)
$$

В модели использованы следующие эмпирические корреляции для PEMFC: расширенная корреляция Антуана для давления насыщенного пара воды и корреляция Springer–Zawodzinski–Gottesfeld для водосодержания мембраны Nafion.

### 4.2. Активность воды

Активность воды на стороне мембраны задается как отношение парциального давления пара к давлению насыщения:

$$
a_w = \frac{p_{H_2O}}{p_{\mathrm{sat}}(T)}
$$

$a_w = 1$ соответствует насыщению, $a_w < 1$ соответствует ненасыщенному пару.

### 4.3. Водосодержание мембраны

Функция `membrane_water_content(activity)` вычисляет

$$
\lambda_{\mathrm{mem}}
$$

где $\lambda_{\mathrm{mem}}$ это число молей воды на 1 моль фиксированных сульфогрупп $\mathrm{SO_3^-}$.  
Водосодержание мембраны рассчитывается по эмпирической корреляции Springer–Zawodzinski–Gottesfeld для мембраны Nafion.

### 4.4. Объемная доля воды в мембране

Функция `membrane_water_volume_fraction(...)` переводит водосодержание в объемную долю воды:

$$
\phi_w =
\frac{\lambda_{\mathrm{mem}} V_w}
{V_{\mathrm{SO_3}} + \lambda_{\mathrm{mem}} V_w}
$$

где:

- $V_w$ это молярный объем воды
- $V_{\mathrm{SO_3}}$ это эффективный удельный объем сухого иономера на 1 моль кислотных центров

Физический смысл: чем больше воды удерживает мембрана, тем выше ее степень набухания и тем сильнее меняются транспортные свойства.

### 4.5. Коэффициент electro-osmotic drag

Функция `electro_osmotic_drag_coefficient(lambda_avg)` задает коэффициент:

$$
n_{\mathrm{drag}} = \frac{2.5}{22} \lambda_{\mathrm{avg}}
$$

Коэффициент электроосмотического переноса определяет, сколько молей воды переносится через мембрану от анода к катоду одним молем протонов

### 4.6. Диффузия воды в мембране

Функция `water_diffusivity_membrane_m2_s(lambda_avg, T_K)` возвращает:
$$
D_w = D_w(\lambda_{\mathrm{avg}}, T)
$$
Протонная проводимость мембраны рассчитывается по кусочно-заданной эмпирической зависимости от среднего водосодержани $\lambda_{\mathrm{avg}}$ температурная поправка задается экспоненциальным множителем по Аррениусу:

$$
D_w \propto \exp \left[
2416 \left(\frac{1}{298.15} - \frac{1}{T}\right)
\right]
$$

Физически это означает:

1. более увлажненная мембрана проводит воду быстрее  
2. с ростом температуры диффузия возрастает


In [4]:
def water_saturation_pressure_Pa(T_K: float) -> float:
    # Давление насыщенного водяного пара, Па.
    # Формула воспроизводит вид зависимости из модели SC_3.
    return 133.32 * 10 ** (
        29.8605
        - 3.1522e3 / T_K
        - 7.3037 * log10(T_K)
        + 2.4247e-9 * T_K
        + 1.809e-6 * T_K * T_K
    )


def membrane_water_activity(p_h2o_Pa: float, T_K: float) -> float:
    # Активность воды как отношение парциального давления пара к давлению насыщения.
    return p_h2o_Pa / water_saturation_pressure_Pa(T_K)


def membrane_water_content(activity: float) -> float:
    # Водосодержание мембраны lambda_mem.
    # Piecewise-зависимость воспроизводит форму из SC_3.
    a = activity
    if a <= 1.0:
        return 0.043 + 17.81 * a - 39.85 * a**2 + 36.0 * a**3
    if a <= 3.0:
        return 14.0 + 1.4 * (a - 1.0)
    return 16.8


def membrane_water_volume_fraction(
    lambda_mem: float,
    ew_g_per_mol: float = EW,
    rho_memb_kg_m3: float = rho_memb,
) -> float:
    # Объёмная доля воды в мембране.
    # 1.8e-5 м^3/моль — молярный объём воды.
    # V_SO3 вычисляется из эквивалентной массы и плотности сухого иономера.
    v_w = 1.8e-5
    v_so3 = (ew_g_per_mol / 1000.0) / rho_memb_kg_m3
    return lambda_mem * v_w / (v_so3 + lambda_mem * v_w)


def electro_osmotic_drag_coefficient(lambda_avg: float) -> float:
    # Коэффициент electro-osmotic drag.
    return 2.5 * lambda_avg / 22.0


def water_diffusivity_membrane_m2_s(lambda_avg: float, T_K: float) -> float:
    # Эффективный коэффициент диффузии воды в мембране, м^2/с.
    # Восстановлен как piecewise-функция по lambda_avg с температурным множителем из SC_3.
    lmb = lambda_avg
    if lmb < 2.0:
        base = lmb * 1.0e-9
    elif lmb <= 3.0:
        base = 1.0e-9 * (1.0 + 2.0 * (lmb - 2.0))
    elif lmb <= 4.5:
        base = 1.0e-9 * (3.0 - 1.67 * (lmb - 3.0))
    else:
        base = 1.25e-9
    return base * exp(2416.0 * (1.0 / 298.15 - 1.0 / T_K))



## 5. Основная функция модели `MEMB`

Функция рассчитывает:

- перенос кислорода от катода к аноду
- перенос азота от катода к аноду
- перенос водорода от анода к катоду
- диффузионный перенос воды от катода к аноду
- электроосмотический перенос воды от анода к катоду
- реакционный расход водорода

Проницаемость мембраны для газов определяется через `phi_w_avg`, а диффузия воды через мембрану использует `lambda_avg`, кусочно-заданные зависимости `D_w` и концентрацию фиксированных сульфогрупп `c_so3`.

## 5.1. Парциальные давления компонентов

Для каждой стороны мембраны используются стандартные соотношения:

$$
p_i = x_i \, p
$$

где:

- $x_i$ это молярная доля компонента
- $p$ это полное давление смеси
- $p_i$ это парциальное давление компонента

Именно разности парциальных давлений дают движущую силу для газового переноса.

## 5.2. Газовый перенос через мембрану

Для `O2`, `N2` и `H2` поток задан в линейном по перепаду давлений виде:

$$
\dot n_i = k_i \, \Delta p_i
$$

где:

- $\dot n_i$ это молярный поток компонента через мембрану
- $k_i$ это эффективный пермеанс мембраны
- $\Delta p_i$ это разность парциальных давлений данного газа по разные стороны мембраны

Проницаемости мембраны зависят от средней объемной доли воды  $\phi_{w,\mathrm{avg}}$, температуры, площади мембраны и толщины:

$$
k_i \sim f_i(\phi_{w,\mathrm{avg}})
\exp\!\left[
\frac{E_i}{R}
\left(
\frac{1}{303} - \frac{1}{T}
\right)
\right]
\frac{A}{l_m}
$$

Физический смысл:

- более тонкая мембрана повышает перенос газов
- большая площадь мембраны повышает перенос газов
- гидратация мембраны влияет на ее газопроницаемость
- температура меняет перенос через экспоненциальный множитель

## 5.3. Диффузионный перенос воды

Диффузионный поток воды в сторону анода рассчитывается как

$$
\dot n_{H_2O,\mathrm{diff \to anode}} =
D_w \, c_{\mathrm{SO_3}}
\frac{\lambda_c - \lambda_a}{l_m} A
$$

где:

- $D_w$ это коэффициент диффузии воды в мембране
- $c_{\mathrm{SO_3}}$ это концентрация фиксированных кислотных центров в мембране
- $\lambda_c$, $\lambda_a$ это гидратация мембраны со стороны катода и анода
- $l_m$ это толщина мембраны
- $A$ это активная площадь

По сути это запись закона Фика через градиент водосодержания мембраны.

## 5.4. Электроосмотический перенос

Поток воды, увлекаемой протонами в сторону катода, считается как

$$
\dot n_{H_2O,\mathrm{drag \to cathode}} =
n_{\mathrm{drag}}
\frac{I \, N_{\mathrm{cell}}}{F}
$$

где:

- $I$ это ток через стек
- $N_{\mathrm{cell}}$ это число ячеек БТЭ
- $F$ это число Фарадея
- $n_{\mathrm{drag}}$ это коэффициент electro-osmotic drag

Физически: чем выше ток, тем интенсивнее перенос протонов и тем сильнее перенос воды к катоду.

## 5.5. Суммарный поток воды

Знак выбран так, что положительное значение соответствует переносу воды в анод:

$$
\dot n_{H_2O,\mathrm{net \to anode}} =
\dot n_{H_2O,\mathrm{diff \to anode}}
-
\dot n_{H_2O,\mathrm{drag \to cathode}}
$$

Общий перенос воды определяется действием двух механизмов:

- диффузия от катода к аноду
- электроосмотический перенос от анода к катоду

## 5.6. Реакционный расход водорода

Расход водорода на аноде рассчитывается по закону Фарадея:

$$
\dot n_{H_2,\mathrm{cons}} = - \frac{I \, N_{\mathrm{cell}}}{2F}
$$

Минус в коде означает расход компонента на анодной стороне.  
Число 2 в знаменателе связано с тем, что на 1 моль \(H_2\) приходится 2 моля электронов.

## 5.7. Принятая знаковая конвенция

Знаки потоков приняты следущими:

- `n_o2_to_anode_mol_s > 0` означает, что `O₂` идет в анод
- `n_n2_to_anode_mol_s > 0` означает, что `N₂` идет в анод
- `n_h2_to_cathode_mol_s > 0` означает, что `H₂` идет в катод
- `n_h2o_net_to_anode_mol_s > 0` означает, что суммарно вода идет в анод


In [1]:
def sc3_membrane_model(
    current_A: float,
    cathode: GasState,
    anode: GasState,
    params: BTEParams | None = None,
) -> SC3Result:
    # Основной расчёт мембранных потоков в SC_3.
    params = params or BTEParams()

    # Проверка корректности состава на обеих сторонах.
    cathode.validate()
    anode.validate()

    # Давление насыщенного пара воды на катоде и аноде, Па.
    p_sat_c = water_saturation_pressure_Pa(cathode.temperature_K)
    p_sat_a = water_saturation_pressure_Pa(anode.temperature_K)

    # Активность воды на катодной и анодной сторонах.
    activity_c = membrane_water_activity(cathode.p_h2o, cathode.temperature_K)
    activity_a = membrane_water_activity(anode.p_h2o, anode.temperature_K)

    # Локальное водосодержание мембраны на двух сторонах.
    lambda_c = membrane_water_content(activity_c)
    lambda_a = membrane_water_content(activity_a)
    lambda_avg = 0.5 * (lambda_c + lambda_a)

    # Объёмная доля воды в мембране.
    phi_c = membrane_water_volume_fraction(lambda_c, params.ew_g_per_mol, params.rho_memb_kg_m3)
    phi_a = membrane_water_volume_fraction(lambda_a, params.ew_g_per_mol, params.rho_memb_kg_m3)
    phi_avg = membrane_water_volume_fraction(lambda_avg, params.ew_g_per_mol, params.rho_memb_kg_m3)

    # Геометрия мембраны.
    area = params.area_m2_total
    lm = params.thickness_m

    # Разности парциальных давлений для crossover, Па.
    # O2 и N2 идут из катода в анод.
    # H2 идёт из анода в катод.
    dp_o2 = cathode.p_o2 - anode.p_o2
    dp_n2 = cathode.p_n2 - anode.p_n2
    dp_h2 = anode.p_h2 - cathode.p_h2

    # Газовые пермеансы мембраны.
    # Коэффициенты и температурные экспоненты заданы в инженерной форме и сопоставлены с Amesim.
    # Для газового crossover используется средняя объёмная доля воды phi_w_avg.
    k_o2 = (
        (0.11 + 1.9 * phi_avg)
        * 1e-14
        * exp(22000.0 / R_UNIV * (1.0 / 303.0 - 1.0 / cathode.temperature_K))
        * area
        / lm
    )
    k_n2 = (
        (0.0295 + 1.21 * phi_avg - 1.93 * phi_avg**2)
        * 1e-14
        * exp(24000.0 / R_UNIV * (1.0 / 303.0 - 1.0 / cathode.temperature_K))
        * area
        / lm
    )
    k_h2 = (
        (0.29 + 2.2 * phi_avg)
        * 1e-14
        * exp(21000.0 / R_UNIV * (1.0 / 303.0 - 1.0 / cathode.temperature_K))
        * area
        / lm
    )

    # Потоки crossover через мембрану, моль/с.
    n_o2 = k_o2 * dp_o2
    n_n2 = k_n2 * dp_n2
    n_h2 = k_h2 * dp_h2

    # Коэффициент electro-osmotic drag.
    n_drag = electro_osmotic_drag_coefficient(lambda_avg)

    # Диффузия воды через мембрану.
    # Для температурного множителя используется температура катодной стороны,
    # как в реконструкции исходной логики SC_3.
    d_w = water_diffusivity_membrane_m2_s(lambda_avg, cathode.temperature_K)
    n_h2o_diff = d_w * params.c_so3_mol_m3 * ((lambda_c - lambda_a) / lm) * area

    # Electro-osmotic drag тянет воду с анода к катоду.
    n_h2o_drag_to_cathode = n_drag * current_A * params.ncell / FARADAY

    # Суммарный поток воды в сторону анода.
    n_h2o_net_to_anode = n_h2o_diff - n_h2o_drag_to_cathode

    # Реакционный расход H2 на аноде.
    n_h2_consumption = -(current_A * params.ncell) / (2.0 * FARADAY)

    return SC3Result(
        n_o2_to_anode_mol_s=n_o2,
        n_n2_to_anode_mol_s=n_n2,
        n_h2_to_cathode_mol_s=n_h2,
        n_h2o_diff_to_anode_mol_s=n_h2o_diff,
        n_h2o_drag_to_cathode_mol_s=n_h2o_drag_to_cathode,
        n_h2o_net_to_anode_mol_s=n_h2o_net_to_anode,
        n_h2_consumption_mol_s=n_h2_consumption,
        lambda_membrane_anode=lambda_a,
        lambda_membrane_cathode=lambda_c,
        phi_w_anode=phi_a,
        phi_w_cathode=phi_c,
        phi_w_avg=phi_avg,
        a_h2o_anode=activity_a,
        a_h2o_cathode=activity_c,
        p_sat_anode_Pa=p_sat_a,
        p_sat_cathode_Pa=p_sat_c,
        n_drag=n_drag,
    )



NameError: name 'GasState' is not defined

## 6. Определение потоков и конценраций примесей в системе рециркуляции

Тут добавлены две группы вспомогательных функций:

- преобразование результата `MEMB` в формат компонентных молярных потоков
- расчет накопления примесей при закрытых клапанах удаления примесей с учетом:
  - относительной влажности на аноде
  - влагоотделителя
  - полного восполнения реакционного расхода `H2` чистым водородом
  - пользовательского стехиометрического числа `lambda`

## 6.1. Перевод потоков к упорядоченному виду

Потоки собираются в порядке:

1. `O2`
2. `N2`
3. `H2O`
4. `H2`

Для водорода выводится суммарный источник на аноде:

$$
\dot n_{H_2,\mathrm{source}} =
\dot n_{H_2,\mathrm{cons}} -
\dot n_{H_2,\mathrm{cross}}
$$

где второй член учитывает утечку водорода через мембрану в катод.

Суммарный массовый поток источника вычисляется как

$$
\dot m_3 = \sum_i \dot n_i M_i
$$

где \(M_i\) это молярные массы соответствующих компонентов.

## 6.2. Относительная влажность на аноде

В функции `relative_humidity_from_molar_fraction(...)` влажность определяется как

$$
humid = \frac{p_{H_2O}}{p_{\mathrm{sat}}(T)}
= \frac{x_{H_2O} \, p}{p_{\mathrm{sat}}(T)}
$$

То есть вход в расчет идет через молярную долю пара, абсолютное давление и температуру.

## 6.3. Баланс водорода при закрытой рециркуляции

В расчете накопления примесей используются такие соотношения:

$$
\dot n_{H_2,\mathrm{makeup}} =
\left|\dot n_{H_2,\mathrm{cons}}\right|
$$

$$
\dot n_{H_2,\mathrm{inlet\ to\ BTE}} =
\lambda \cdot \left|\dot n_{H_2,\mathrm{cons}}\right|
$$

$$
\dot n_{H_2,\mathrm{required\ from\ recirc}} =
\dot n_{H_2,\mathrm{inlet\ to\ BTE}} -
\dot n_{H_2,\mathrm{makeup}}
$$

Их физический смысл:

- чистый подпиточный `H2` полностью закрывает реакционный расход
- входной расход в БТЭ задается через целевое стехиометрическое число \(\lambda\)
- недостающая часть должна прийти из рециркуляции

## 6.4. Учет влагоотделителя

Вода после влагоотделителя задается через относительную влажность:

$$
humid_{out,raw} = eff \cdot humid_{in}
$$

После этого влажность ограничивается сверху насыщением:

$$
humid_{out} = \min(1,\max(0, humid_{out,raw}))
$$

а новая молярная доля пара в газовой фазе вычисляется как

$$
x_{H_2O,\mathrm{after\ separator}} =
humid_{out}\frac{p_{\mathrm{sat}}(T)}{p}
$$

То есть влагоотделитель в этой постановке не "вычитает моли напрямую", а меняет влажность газовой фазы, после чего через это пересчитывается доля пара.

## 6.5. Потоки примесей, остающихся в контуре

После влагоотделителя в контуре накапливаются:

$$
\dot n_{O_2,\mathrm{acc}} = \dot n_{O_2,\mathrm{to\ anode}}
$$

$$
\dot n_{N_2,\mathrm{acc}} = \dot n_{N_2,\mathrm{to\ anode}}
$$

$$
\dot n_{H_2O,\mathrm{acc}} =
\dot n_{H_2O,\mathrm{net\ to\ anode}}
\cdot
\mathrm{water\_remaining\_fraction}
$$

Суммарная скорость накопления примесей:

$$
\dot n_{\mathrm{imp,tot}} =
\dot n_{O_2,\mathrm{acc}} +
\dot n_{N_2,\mathrm{acc}} +
\dot n_{H_2O,\mathrm{acc}}
$$


In [6]:
COMPONENT_MOLAR_MASSES_G_MOL = {
    "o2": M_O2_G_MOL,
    "n2": M_N2_G_MOL,
    "h2o": M_H2O_G_MOL,
    "h2": M_H2_G_MOL,
}


def sc3_amesim_flux_dataframe(result: SC3Result) -> pd.DataFrame:
    # Потоки компонентов в формате, близком к панели Amesim.
    h2_source_to_anode = result.n_h2_consumption_mol_s - result.n_h2_to_cathode_mol_s

    rows = [
        ("effective molar flux (1)", result.n_o2_to_anode_mol_s, "mol/s", "O2 в анод"),
        ("effective molar flux (2)", result.n_n2_to_anode_mol_s, "mol/s", "N2 в анод"),
        ("effective molar flux (3)", result.n_h2o_net_to_anode_mol_s, "mol/s", "H2O в анод"),
        ("effective molar flux (4)", h2_source_to_anode, "mol/s", "H2 на аноде: реакция + crossover"),
    ]

    mass_flow_port3_g_s = (
        result.n_o2_to_anode_mol_s * COMPONENT_MOLAR_MASSES_G_MOL["o2"]
        + result.n_n2_to_anode_mol_s * COMPONENT_MOLAR_MASSES_G_MOL["n2"]
        + result.n_h2o_net_to_anode_mol_s * COMPONENT_MOLAR_MASSES_G_MOL["h2o"]
        + h2_source_to_anode * COMPONENT_MOLAR_MASSES_G_MOL["h2"]
    )
    rows.append(("mass flow rate at port 3", mass_flow_port3_g_s, "g/s", "Суммарный массовый поток источника"))

    return pd.DataFrame(rows, columns=["title", "value", "unit", "meaning"])


def relative_humidity_from_molar_fraction(
    x_h2o: float,
    pressure_Pa: float,
    temperature_K: float,
) -> tuple[float, float]:
    # Относительная влажность и давление насыщенного пара.
    p_sat = water_saturation_pressure_Pa(temperature_K)
    humidity = x_h2o * pressure_Pa / p_sat
    return humidity, p_sat


def closed_recirc_impurity_accumulation_dataframe(
    result: SC3Result,
    anode: GasState,
    lambda_stoich: float,
    water_separator_eff_percent: float,
    accumulation_time_s: float,
) -> pd.DataFrame:
    # Баланс накопления примесей при закрытой рециркуляции.
    eff_fraction = max(0.0, water_separator_eff_percent / 100.0)

    # Реакционный расход водорода в положительном виде.
    n_h2_consumption_abs = -result.n_h2_consumption_mol_s

    # Полное восполнение расхода H2 чистым водородом.
    n_h2_makeup_pure_mol_s = n_h2_consumption_abs

    # Требуемый расход на входе в БТЭ по заданному lambda.
    n_h2_inlet_to_bte_mol_s = lambda_stoich * n_h2_consumption_abs

    # Требуемая доля рециркуляции для достижения заданного входного расхода.
    n_h2_required_from_recirc_mol_s = max(
        0.0, n_h2_inlet_to_bte_mol_s - n_h2_makeup_pure_mol_s
    )

    # Относительная влажность на аноде до влагоотделителя.
    humidity_in, p_sat_anode = relative_humidity_from_molar_fraction(
        x_h2o=anode.x_h2o,
        pressure_Pa=anode.pressure_Pa,
        temperature_K=anode.temperature_K,
    )

    # Формула из постановки пользователя.
    humidity_out_raw = eff_fraction * humidity_in

    # Газовая фаза после влагоотделителя ограничивается насыщением.
    humidity_out = min(1.0, max(0.0, humidity_out_raw))
    x_h2o_after_separator = min(
        0.999999, humidity_out * p_sat_anode / anode.pressure_Pa
    )

    # Доля воды, остающаяся в контуре после влагоотделителя.
    water_remaining_fraction = 0.0
    if abs(humidity_in) > 1e-16:
        water_remaining_fraction = humidity_out / humidity_in

    # Потоки примесей, остающиеся в контуре после влагоотделителя.
    n_o2_accumulation_mol_s = result.n_o2_to_anode_mol_s
    n_n2_accumulation_mol_s = result.n_n2_to_anode_mol_s
    n_h2o_after_separator_mol_s = (
        result.n_h2o_net_to_anode_mol_s * water_remaining_fraction
    )
    n_h2o_removed_mol_s = (
        result.n_h2o_net_to_anode_mol_s - n_h2o_after_separator_mol_s
    )

    n_total_impurity_accumulation_mol_s = (
        n_o2_accumulation_mol_s
        + n_n2_accumulation_mol_s
        + n_h2o_after_separator_mol_s
    )

    rows = [
        ("n_h2_consumption_abs_mol_s", n_h2_consumption_abs, "mol/s", "Положительный расход H2 на реакцию в БТЭ"),
        ("n_h2_makeup_pure_mol_s", n_h2_makeup_pure_mol_s, "mol/s", "Чистый H2, полностью компенсирующий реакционный расход"),
        ("n_h2_inlet_to_bte_mol_s", n_h2_inlet_to_bte_mol_s, "mol/s", "Требуемый расход на входе в БТЭ: dm1 = lambda * dm_H2"),
        ("n_h2_required_from_recirc_mol_s", n_h2_required_from_recirc_mol_s, "mol/s", "Требуемая доля H2 из рециркуляции"),
        ("humidity_in", humidity_in, "-", "Относительная влажность на аноде до влагоотделителя"),
        ("humidity_out_raw", humidity_out_raw, "-", "Влажность по формуле humid_out = eff * humid_in"),
        ("humidity_out", humidity_out, "-", "Влажность газовой фазы после ограничения насыщением"),
        ("x_h2o_after_separator", x_h2o_after_separator, "-", "Молярная доля H2O после влагоотделителя"),
        ("n_h2o_removed_mol_s", n_h2o_removed_mol_s, "mol/s", "Удалённая вода во влагоотделителе"),
        ("n_o2_accumulation_mol_s", n_o2_accumulation_mol_s, "mol/s", "Скорость накопления O2 в контуре"),
        ("n_n2_accumulation_mol_s", n_n2_accumulation_mol_s, "mol/s", "Скорость накопления N2 в контуре"),
        ("n_h2o_accumulation_mol_s", n_h2o_after_separator_mol_s, "mol/s", "Скорость накопления H2O после влагоотделителя"),
        ("n_total_impurity_accumulation_mol_s", n_total_impurity_accumulation_mol_s, "mol/s", "Суммарная скорость накопления примесей"),
        ("n_o2_accumulated_over_time_mol", n_o2_accumulation_mol_s * accumulation_time_s, "mol", "Накопление O2 за выбранный горизонт"),
        ("n_n2_accumulated_over_time_mol", n_n2_accumulation_mol_s * accumulation_time_s, "mol", "Накопление N2 за выбранный горизонт"),
        ("n_h2o_accumulated_over_time_mol", n_h2o_after_separator_mol_s * accumulation_time_s, "mol", "Накопление H2O за выбранный горизонт"),
        ("n_total_impurity_over_time_mol", n_total_impurity_accumulation_mol_s * accumulation_time_s, "mol", "Суммарное накопление примесей за выбранный горизонт"),
    ]

    return pd.DataFrame(rows, columns=["variable", "value", "unit", "meaning"])


## 7. Состояния анода и катода

В этой части из верхней ячейки собираются состояния смеси для анода и катода:

- температура смеси
- абсолютное давление смеси
- молярные доли компонентов `H₂`, `N₂`, `H₂O`, `O₂`

Так легче менять режим расчёта без правки внутренней логики модели.


In [7]:
def make_amesim_anode_state(
    temperature_K: float = ANODE_T_K,
    pressure_Pa: float = ANODE_P_PA,
    x_h2: float = ANODE_X_H2,
    x_n2: float = ANODE_X_N2,
    x_h2o: float = ANODE_X_H2O,
    x_o2: float = ANODE_X_O2,
) -> GasState:
    # Анодное состояние смеси из входных данных Amesim.
    return GasState(
        temperature_K=temperature_K,
        pressure_Pa=pressure_Pa,
        x_h2=x_h2,
        x_n2=x_n2,
        x_h2o=x_h2o,
        x_o2=x_o2,
    )


def make_amesim_cathode_state(
    temperature_K: float = CATHODE_T_K,
    pressure_Pa: float = CATHODE_P_PA,
    x_h2: float = CATHODE_X_H2,
    x_n2: float = CATHODE_X_N2,
    x_h2o: float = CATHODE_X_H2O,
    x_o2: float = CATHODE_X_O2,
) -> GasState:
    # Катодное состояние смеси из входных данных Amesim.
    return GasState(
        temperature_K=temperature_K,
        pressure_Pa=pressure_Pa,
        x_h2=x_h2,
        x_n2=x_n2,
        x_h2o=x_h2o,
        x_o2=x_o2,
    )


## 8. Пример расчёта

Здесь собираются параметры, состояния сторон мембраны и выполняется один расчёт для заданного тока.
Результат выводится таблицей.


In [8]:
# Создаём объект параметров БТЭ.
params = BTEParams()

# Анод и катод собираются из верхней ячейки с входными данными Amesim.
anode = make_amesim_anode_state()
cathode = make_amesim_cathode_state()

# Выполняем расчёт SC_3 для выбранного тока, А.
result = sc3_membrane_model(
    current_A=CURRENT_A,
    cathode=cathode,
    anode=anode,
    params=params,
)

# Табличный просмотр результата.
result_to_dataframe(result)


,variable,value
0,n_o2_to_anode_mol_s,0.000000
1,n_n2_to_anode_mol_s,0.000100
2,n_h2_to_cathode_mol_s,0.000502
3,n_h2o_diff_to_anode_mol_s,1.201510
4,n_h2o_drag_to_cathode_mol_s,1.198631
5,n_h2o_net_to_anode_mol_s,0.002879
6,n_h2_consumption_mol_s,-0.388660
7,lambda_membrane_anode,12.671408
8,lambda_membrane_cathode,14.467874
9,phi_w_anode,0.293137


## 9. Молярные потоки компонентов

В этой ячейке выводятся компонентные молярные потоки в следующем порядке:

1. `O2`
2. `N2`
3. `H2O`
4. `H2`

Для компонента `H2` выводится суммарный источник на аноде:

$$
\dot n_{H_2,\mathrm{source}} =
\dot n_{H_2,\mathrm{cons}} - \dot n_{H_2,\mathrm{cross}}
$$

Также рассчитывается суммарный балланс массовых расходов.


In [9]:
sc3_amesim_flux_dataframe(result)


,title,value,unit,meaning
0,effective molar flux (1),0.000000,mol/s,O2 в анод
1,effective molar flux (2),0.000100,mol/s,N2 в анод
2,effective molar flux (3),0.002879,mol/s,H2O в анод
3,effective molar flux (4),-0.389162,mol/s,H2 на аноде: реакция + crossover
4,mass flow rate at port 3,-0.729890,g/s,Суммарный массовый поток источника


## 10. Быстрый просмотр ключевых потоков

Удобна для экспресс-проверки знаков и порядков величин без просмотра всей таблицы.
Порядок вывода:

1. `O2` в анод
2. `N2` в анод
3. `H2O` в анод
4. `H2` как суммарный источник на аноде


In [10]:
(
    result.n_o2_to_anode_mol_s,
    result.n_n2_to_anode_mol_s,
    result.n_h2o_net_to_anode_mol_s,
    result.n_h2_consumption_mol_s - result.n_h2_to_cathode_mol_s,
)


(0.0, 0.00010011940788173698, 0.0028785171112983843, -0.38916224310991604)

## 11. Накопление примесей при закрытых клапанах удаления примесей

В этой ячейке считается скорость накопления примесей в контуре при закрытых клапанах удаления примесей.

Учтено:

1. влагоотделитель через снижение относительной влажности
2. полное восполнение реакционного расхода `H2` чистым водородом
3. требуемый расход на входе в БТЭ по формуле `dm1 = lambda * dm_H2`
4. пользовательское значение `lambda`
5. накопление за выбранный временной горизонт `ACCUMULATION_TIME_S`

## 11.1. Что именно считается

Моделируется динамическая модель замкнутого контура анода топливного элемента.  
Рассчитываются **квазистационарные скорости накопления** примесей и пересчитываются в количество за заданный интервал времени.

Для выбранного горизонта \(t_{\mathrm{acc}}\):

$$
N_i(t_{\mathrm{acc}}) = \dot n_i \, t_{\mathrm{acc}}
$$

где:

- $N_i$ это накопленное количество компонента, моль
- $\dot n_i$ это скорость накопления данного компонента, моль/с

В таблице ниже отдельно выводятся:

- `O2`
- `N2`
- `H2O` после влагоотделителя
- суммарное накопление примесей

## 11.2. Как читать результат

Этот блок удобен для быстрых инженерных ответов на вопросы вида:

- какой компонент загрязняет анодный контур быстрее всего
- насколько сильно помогает влагоотделитель
- какую часть входного расхода нужно добирать из рециркуляции
- какой объём примесей накопится за условные 10, 30, 60 или 300 секунд


In [11]:
closed_recirc_impurity_accumulation_dataframe(
    result=result,
    anode=anode,
    lambda_stoich=ANODE_STOICH_LAMBDA,
    water_separator_eff_percent=WATER_SEPARATOR_EFF_PERCENT,
    accumulation_time_s=ACCUMULATION_TIME_S,
)


,variable,value,unit,meaning
0,n_h2_consumption_abs_mol_s,0.388660,mol/s,Положительный расход H2 на реакцию в БТЭ
1,n_h2_makeup_pure_mol_s,0.388660,mol/s,"Чистый H2, полностью компенсирующий реакционны..."
2,n_h2_inlet_to_bte_mol_s,0.582990,mol/s,Требуемый расход на входе в БТЭ: dm1 = lambda ...
3,n_h2_required_from_recirc_mol_s,0.194330,mol/s,Требуемая доля H2 из рециркуляции
4,humidity_in,0.969794,-,Относительная влажность на аноде до влагоотдел...
5,humidity_out_raw,0.901909,-,Влажность по формуле humid_out = eff * humid_in
6,humidity_out,0.901909,-,Влажность газовой фазы после ограничения насыщ...
7,x_h2o_after_separator,0.017551,-,Молярная доля H2O после влагоотделителя
8,n_h2o_removed_mol_s,0.000201,mol/s,Удалённая вода во влагоотделителе
9,n_o2_accumulation_mol_s,0.000000,mol/s,Скорость накопления O2 в контуре


## 12. Литература и ссылки

Ниже приведены основные источники, на которые опирается физический смысл уравнений в этом ноутбуке.

### Базовые статьи по PEMFC и мембране

1. **Springer, T. E.; Zawodzinski, T. A.; Gottesfeld, S. (1991). _Polymer Electrolyte Fuel Cell Model_. Journal of The Electrochemical Society, 138(8), 2334-2342.**  
   DOI: [10.1149/1.2085971](https://doi.org/10.1149/1.2085971)  
   Зачем нужен в этом ноутбуке: классическая основа для связи гидратации мембраны, water drag и переноса воды в PEMFC.

2. **Bernardi, D. M.; Verbrugge, M. W. (1992). _A Mathematical Model of the Solid-Polymer-Electrolyte Fuel Cell_. Journal of The Electrochemical Society, 139(9), 2477-2491.**  
   DOI: [10.1149/1.2221251](https://doi.org/10.1149/1.2221251)  
   Зачем нужен в этом ноутбуке: общая физика стационарной модели PEMFC, балансы компонентов и связь токов с расходами реагентов.

3. **Zawodzinski, T. A. Jr. et al. (1993). _Water Uptake by and Transport Through Nafion® 117 Membranes_. Journal of The Electrochemical Society, 140(4), 1041-1047.**  
   DOI: [10.1149/1.2056194](https://doi.org/10.1149/1.2056194)  
   Зачем нужен в этом ноутбуке: водосодержание мембраны, зависимость от активности воды, транспорт воды в Nafion.

4. **Motupally, S.; Becker, A. J.; Weidner, J. W. (2000). _Diffusion of Water in Nafion 115 Membranes_. Journal of The Electrochemical Society, 147(9), 3171-3177.**  
   DOI: [10.1149/1.1393879](https://doi.org/10.1149/1.1393879)  
   Зачем нужен в этом ноутбуке: интерпретация коэффициента диффузии воды и температурно-влажностной чувствительности \(D_w\).

5. **Weber, A. Z.; Newman, J. (2004). _Modeling Transport in Polymer-Electrolyte Fuel Cells_. Chemical Reviews, 104(10), 4679-4726.**  
   DOI: [10.1021/cr020729l](https://doi.org/10.1021/cr020729l)  
   Зачем нужен в этом ноутбуке: обзорная рамка по массопереносу, мембране, воде, газам и ограничениям упрощённых моделей.

6. **Weber, A. Z.; Newman, J. (2004). _Transport in Polymer-Electrolyte Membranes. II. Mathematical Model_. Journal of The Electrochemical Society, 151(2), A311-A325.**  
   DOI: [10.1149/1.1639158](https://doi.org/10.1149/1.1639158)  
   Зачем нужен в этом ноутбуке: математическая постановка переноса в полимерной мембране и интерпретация эффективных коэффициентов.

### Источники по gas crossover

7. **Kocha, S. S.; Yang, J. D.; Yi, J. S. (2006). _Characterization of Gas Crossover and Its Implications in PEM Fuel Cells_. AIChE Journal, 52(5), 1916-1925.**  
   DOI: [10.1002/aic.10780](https://doi.org/10.1002/aic.10780)  
   Зачем нужен в этом ноутбуке: физический смысл газового crossover и влияние проницаемости мембраны на анодный контур.

### Источник по давлению насыщения воды

8. **NIST Chemistry WebBook, Water, Antoine Equation Parameters.**  
   Ссылка: [NIST WebBook: Water](https://webbook.nist.gov/cgi/cbook.cgi?ID=C7732185&Mask=4&Type=ANTOINE)  
   Зачем нужен в этом ноутбуке: справочная база для проверки корреляций давления насыщенного пара воды.

## Карта "уравнение → источник"

- `a_w = p_H2O / p_sat(T)` → Zawodzinski 1993, Weber & Newman 2004  
- `lambda_mem = f(a_w)` → Springer 1991, Zawodzinski 1993  
- `n_drag = f(lambda)` → Springer 1991  
- `D_w = f(lambda, T)` → Motupally 2000, Weber & Newman 2004  
- `n_i = k_i * Δp_i` для gas crossover → инженерная форма в текущем ноутбуке, физическая интерпретация опирается на Kocha 2006 и общие PEMFC-модели  
- `n_H2,cons = -I Ncell / (2F)` → стандартный закон Фарадея, применяемый во всех PEMFC-моделях  
- баланс закрытой рециркуляции и накопления примесей → инженерная постановка для данного контура, совместимая с общими компонентными балансами Bernardi 1992 и обзором Weber & Newman 2004
